# Subgraph verification: does RemoteGraph actually control subgraphs?

`researcher`/`coder`/`reviewer` are flat ReAct agents, so they can't prove whether
`RemoteGraph` actually gives central control *inside* a remote graph that contains
a subgraph -- only that it can call the graph as a whole. This notebook deploys a
small, deterministic, LLM-free graph (`agents/subgraph_demo`) that has a real
subgraph node, and checks three things against a live backend:

1. `stream_subgraphs=True` surfaces events from *inside* the subgraph, not just the
   subgraph node's aggregate result.
2. `interrupt_before=["inner"]` actually pauses the run *before* the subgraph node
   runs (`state.next == ("inner",)`), and resuming continues correctly.
3. The same behavior holds through the actual `RemoteGraph` class (what
   `agents/supervisor/graph.py` uses), not just the raw `langgraph_sdk` client.

Backend: `langgraph-platform` (`langgraph dev`, no Docker, no LLM calls needed).

In [1]:
import sys

sys.path.insert(0, "/Users/jyje/repo/jyje/pilot-langchain-remotegraph")

from remotegraph.backends.langgraph_platform import LangGraphPlatformBackend

backend = LangGraphPlatformBackend()
backend.deploy({
    "researcher": "agents/researcher/graph.py:graph",
    "coder": "agents/coder/graph.py:graph",
    "reviewer": "agents/reviewer/graph.py:graph",
    "subgraph_demo": "agents/subgraph_demo/graph.py:graph",
})
backend.up()
print(backend.status())


running


## 1. `stream_subgraphs=True`: can we see inside the subgraph?

`agents/subgraph_demo/graph.py` is `prepare -> inner (a compiled subgraph that
uppercases text) -> finalize`. Without `stream_subgraphs=True` you'd only ever see
`inner`'s aggregate result. With it, the subgraph's own internal node update should
appear as a separately namespaced event.

In [2]:
import time

from langgraph_sdk import get_sync_client

client = get_sync_client(url=backend.base_url)

for _attempt in range(10):
    try:
        client.assistants.search()
        break
    except Exception:
        time.sleep(2)

thread = client.threads.create()
for chunk in client.runs.stream(
    thread["thread_id"],
    "subgraph_demo",
    input={"text": "hello"},
    stream_mode=["updates"],
    stream_subgraphs=True,
):
    print(chunk.event, "->", chunk.data)


metadata -> {'run_id': '019eebfe-b295-79b3-855d-04f44de5bd82', 'attempt': 1}
updates -> {'prepare': {'text': '[prepared] hello'}}
updates|inner:1e8055ec-dce0-2da1-0f28-acc26500cfee -> {'shout': {'text': '[PREPARED] HELLO'}}
updates -> {'inner': {'text': '[PREPARED] HELLO'}}
updates -> {'finalize': {'text': '[PREPARED] HELLO [finalized]'}}


## 2. `interrupt_before=["inner"]`: does it actually pause before the subgraph?

If subgraph control is real, the run should stop with `state.next == ("inner",)`
*before* the subgraph's `shout` node ever executes -- `state.values["text"]` should
still be lowercase, untouched by the subgraph.

In [3]:
thread2 = client.threads.create()
tid = thread2["thread_id"]

for chunk in client.runs.stream(
    tid,
    "subgraph_demo",
    input={"text": "world"},
    stream_mode=["updates"],
    interrupt_before=["inner"],
):
    print(chunk.event, "->", chunk.data)

state = client.threads.get_state(tid)
print()
print("next:", state["next"])
print("values:", state["values"])
assert state["next"] == ["inner"]
assert state["values"]["text"] == "[prepared] world"  # subgraph hasn't run yet


metadata -> {'run_id': '019eebfe-b676-7d60-a44a-904d7b0cfc07', 'attempt': 1}
updates -> {'prepare': {'text': '[prepared] world'}}
updates -> {'__interrupt__': []}

next: ['inner']
values: {'text': '[prepared] world'}


## 3. Resume past the interrupt -- does it continue into the subgraph correctly?

In [4]:
for chunk in client.runs.stream(tid, "subgraph_demo", input=None, stream_mode=["updates"]):
    print(chunk.event, "->", chunk.data)

final_state = client.threads.get_state(tid)
print()
print("next:", final_state["next"])
print("values:", final_state["values"])
assert final_state["values"]["text"] == "[PREPARED] WORLD [finalized]"


metadata -> {'run_id': '019eebfe-ba6f-74c3-b692-9d37ea461cac', 'attempt': 1}
updates -> {'inner': {'text': '[PREPARED] WORLD'}}
updates -> {'finalize': {'text': '[PREPARED] WORLD [finalized]'}}

next: []
values: {'text': '[PREPARED] WORLD [finalized]'}


## 4. Same thing through `RemoteGraph` itself

This is the class `agents/supervisor/graph.py` actually uses -- not the raw SDK
client. If this works the same way, the supervisor pattern genuinely supports
subgraph-level control, not just whole-graph calls.

In [5]:
import uuid

from langgraph.pregel.remote import RemoteGraph

remote = RemoteGraph("subgraph_demo", url=backend.base_url)
config = {"configurable": {"thread_id": str(uuid.uuid4())}}

paused = remote.invoke({"text": "via RemoteGraph"}, config=config, interrupt_before=["inner"])
print("paused result:", paused)

state = remote.get_state(config)
print("next:", state.next)
print("values:", state.values)
assert state.next == ("inner",)

final = remote.invoke(None, config=config)
print("final:", final)
assert final["text"] == "[PREPARED] VIA REMOTEGRAPH [finalized]"


paused result: {'text': '[prepared] via RemoteGraph'}
next: ('inner',)
values: {'text': '[prepared] via RemoteGraph'}


final: {'text': '[PREPARED] VIA REMOTEGRAPH [finalized]'}


## 5. Teardown

In [6]:
backend.down()
print(backend.status())


stopped
